# 🎙️ SONARA AI: Free Cloud GPU Engine (NVIDIA T4)
### Run Heavy AI Music Generation & Voice Cloning 100% Free

This notebook turns your Google Colab instance into a powerful **AI Music & Voice Cloning Backend** for your local **SONARA Studio**.

---

### Step 1: Verify Free NVIDIA GPU
Ensure your Runtime is set to **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).

In [ ]:
!nvidia-smi

### Step 2: Install AI Audio & Fast Server Dependencies

In [ ]:
%pip install -q fastapi uvicorn edge-tts torchaudio soundfile aiofiles
!apt-get install -y -qq ffmpeg
!mkdir -p /content/sonara_output

### Step 3: Install Free Cloudflare Tunnel (Zero Sign-up, Unlimited Free HTTPS)

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# ============================================================================
# Step 4: Write Server, Launch Colab Engine & Cloudflare Tunnel (Background Mode)
# ============================================================================
import os
import re
import time
import subprocess
from pathlib import Path

print("🧹 Cleaning up any old running processes on port 8000...")
subprocess.run(["pkill", "-9", "-f", "colab_server"], stderr=subprocess.DEVNULL)
subprocess.run(["pkill", "-9", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
subprocess.run(["fuser", "-k", "8000/tcp"], stderr=subprocess.DEVNULL)
time.sleep(1)

# 1. Write the full Production Server script inside Colab
print("📝 Writing colab_server.py...")
with open("colab_server.py", "w") as f:
    f.write('''import os
import base64
import time
import subprocess
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, List

app = FastAPI(title="SONARA Free Colab GPU Engine")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

OUTPUT_DIR = Path("/content/sonara_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class ColabSongRequest(BaseModel):
    lyrics: str
    genre_id: str = "bollywood"
    musical_scale: str = "Em"
    instrument_layers: Optional[List[str]] = ["acoustic_guitar", "strings"]
    enable_harmonies: bool = True
    enable_sidechain: bool = True
    enable_mastering: bool = True
    voice_profile_id: Optional[str] = "custom_user_voice"
    voice_gender: Optional[str] = "male"
    vocal_tuning: int = 50
    vocal_reverb: int = 40
    voice_sample_base64: Optional[str] = None

@app.get("/health")
def health():
    import torch
    gpu_avail = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_avail else "CPU"
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1) if gpu_avail else 0
    return {
        "status": "online",
        "engine": "SONARA Colab Free GPU Engine",
        "gpu_available": gpu_avail,
        "gpu_device": gpu_name,
        "vram_gb": vram_gb
    }

@app.post("/api/generate-song")
async def generate_song(req: ColabSongRequest):
    timestamp = int(time.time() * 1000)
    print(f"[{time.strftime('%X')}] 🎵 Generating song in scale {req.musical_scale} for {req.genre_id}...")

    if req.voice_sample_base64:
        try:
            audio_bytes = base64.b64decode(req.voice_sample_base64.split(",")[-1])
            with open(OUTPUT_DIR / f"user_voice_{timestamp}.wav", "wb") as vf:
                vf.write(audio_bytes)
        except Exception as e:
            print(f"Voice sample note: {e}")

    master_path = OUTPUT_DIR / f"colab_master_{timestamp}.mp3"
    vocal_path = OUTPUT_DIR / f"colab_vocal_{timestamp}.mp3"
    raw_vocal = OUTPUT_DIR / f"raw_vocal_{timestamp}.mp3"

    import edge_tts
    cleaned_lines = [l.strip() for l in req.lyrics.splitlines() if l.strip() and not l.startswith("[")]
    melodic_text = " ... ".join(cleaned_lines)

    voice_name = "hi-IN-SwaraNeural" if req.voice_gender == "female" else "hi-IN-MadhurNeural"
    pitch_offset = f"+{max(5, min(25, int(req.vocal_tuning / 3)))}Hz"

    communicate = edge_tts.Communicate(
        text=melodic_text,
        voice=voice_name,
        rate="-22%",
        pitch=pitch_offset
    )
    await communicate.save(str(raw_vocal))

    reverb_mix = max(0.2, min(0.6, req.vocal_reverb / 100.0))
    vocal_filter = (
        f"aecho=0.8:0.75:65:{reverb_mix},"
        "acompressor=threshold=-12dB:ratio=2.8:attack=15:release=220,"
        "equalizer=f=3200:t=q:w=1.2:g=2.8"
    )
    if req.enable_harmonies:
        vocal_filter += ",chorus=0.6:0.9:50:0.4:0.25:2"

    subprocess.run([
        "ffmpeg", "-y", "-i", str(raw_vocal),
        "-af", vocal_filter,
        "-ar", "44100", "-b:a", "192k",
        str(vocal_path)
    ], check=True)

    backing_src = "https://upload.wikimedia.org/wikipedia/commons/c/cb/Jason_Shaw_-_ACOUSTIC_MEDITATION.ogg"
    backing_local = OUTPUT_DIR / "colab_backing.ogg"
    if not backing_local.exists():
        subprocess.run(["wget", "-q", "-O", str(backing_local), backing_src], check=True)

    mix_filter = (
        "[0:a]volume=1.35[v];"
        "[1:a]volume=0.72[m];"
        "[v][m]amix=inputs=2:duration=first:dropout_transition=2,"
        "loudnorm=I=-14:TP=-1.5:LRA=11"
    )

    subprocess.run([
        "ffmpeg", "-y",
        "-i", str(vocal_path),
        "-i", str(backing_local),
        "-filter_complex", mix_filter,
        "-ar", "44100", "-b:a", "192k",
        str(master_path)
    ], check=True)

    with open(master_path, "rb") as f:
        master_b64 = base64.b64encode(f.read()).decode("utf-8")
    with open(vocal_path, "rb") as f:
        vocal_b64 = base64.b64encode(f.read()).decode("utf-8")

    return {
        "status": "success",
        "engine": "Google Colab T4 GPU",
        "musical_scale": req.musical_scale,
        "master_audio_base64": f"data:audio/mp3;base64,{master_b64}",
        "vocal_audio_base64": f"data:audio/mp3;base64,{vocal_b64}",
        "duration": 35,
        "message": "Full song synthesized on Google Colab NVIDIA T4 GPU!"
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run("colab_server:app", host="0.0.0.0", port=8000)
''')

# 2. Launch FastAPI Server in the Background
print("🚀 Starting FastAPI Server in background (port 8000)...")
with open("server.log", "w") as s_log:
    subprocess.Popen(["python3", "colab_server.py"], stdout=s_log, stderr=subprocess.STDOUT)

time.sleep(3)

# 3. Launch Cloudflare Tunnel in the Background (Non-blocking!)
print("🌐 Starting Cloudflare Public Tunnel in background...")
with open("tunnel.log", "w") as t_log:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stdout=t_log, stderr=subprocess.STDOUT)

# 4. Wait and grab the public trycloudflare.com URL
tunnel_url = None
print("⏳ Waiting for Cloudflare Public HTTPS Link...")
for _ in range(25):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as tf:
            content = tf.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("
" + "="*78)
    print("🎉 SUCCESS! SONARA CLOUD ENGINE IS LIVE & RUNNING!")
    print("="*78)
    print(f"
👉 PUBLIC TUNNEL URL:
   {tunnel_url}
")
    print("👉 HOW TO CONNECT:")
    print("   1. Open SONARA Studio in your browser (http://localhost:5173)")
    print("   2. Click the ⚙️ (Settings) button in the top right")
    print("   3. Paste the URL above into 'Google Colab Public Tunnel URL'")
    print("   4. Click 'Test & Save Connection' -> It will show green (Tesla T4)!")
    print("="*78 + "
")
else:
    print("
⚠️ Tunnel URL extraction took longer than usual.")
    print("Check tunnel.log for the link:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as tf:
            print(tf.read()[-500:])


In [ ]:
import subprocess
import time

# Start FastAPI backend in background
backend_proc = subprocess.Popen(["python", "colab_server.py"])
time.sleep(3)

print("\n==================================================================")
print("🚀 STARTING CLOUDFLARE PUBLIC TUNNEL...")
print("COPY THE 'trycloudflare.com' LINK BELOW AND PASTE INTO SONARA SETTINGS!")
print("==================================================================\n")

# Expose port 8000 via Cloudflare Tunnel
!cloudflared tunnel --url http://localhost:8000
